# Algoritmo ECLAT

Notebook elaborado siguiendo la rúbrica de la asignatura (ver `../../rubrica.md`).

**Autor:** Grupo E 
**Fecha:** 13/06/2026

## 1. Descripción

**ECLAT** (*Equivalence Class Clustering and bottom-up Lattice Traversal*) es un algoritmo de **minería de reglas de asociación** y descubrimiento de **itemsets frecuentes** en bases de datos transaccionales. Fue propuesto por Zaki, Parthasarathy, Ogihara y Li en 1997.

Su objetivo es encontrar relaciones del tipo:

> *Si un cliente compra pan y mantequilla, entonces es probable que también compre leche.*

Formalmente, dada una base de transacciones $D = \{T_1, T_2, \dots, T_n\}$ donde cada $T_i$ es un conjunto de ítems, ECLAT busca todos los itemsets $X \subseteq I$ cuyo **soporte** supere un umbral mínimo, y a partir de ellos genera **reglas de asociación** $X \Rightarrow Y$ con métricas de calidad (confianza, lift, etc.).

La característica distintiva de ECLAT es el uso de una **representación vertical de datos** mediante TID-sets:

- **Representación vertical:** cada ítem apunta al conjunto de IDs de transacciones que lo contienen (TID-set).
- **Búsqueda en profundidad (DFS):** explora el espacio de itemsets de forma recursiva.
- **Cálculo eficiente:** el soporte se obtiene por intersecciones de TID-sets: $\text{sup}(X \cup Y) = \frac{|\text{TID}(X) \cap \text{TID}(Y)|}{|D|}$

Esta estrategia proporciona:
1. Menor número de pasadas sobre la base de datos.
2. Menor consumo de I/O.
3. Mejor localidad de caché en memoria.

### Casos de uso típicos
- Análisis de la cesta de la compra (*market basket analysis*).
- Recomendación de productos.
- Detección de patrones en logs web.
- Análisis de patrones en datos médicos (co-morbilidades).
- Bioinformática (co-ocurrencia de genes/proteínas).

## 2. Bibtex y Referencias

### BibTeX
```bibtex
@inproceedings{zaki1997eclat,
  title     = {New Algorithms for Fast Discovery of Association Rules},
  author    = {Zaki, Mohammed J. and Parthasarathy, Srinivasan and Ogihara, Mitsunori and Li, Wei},
  booktitle = {Proceedings of the 3rd International Conference on Knowledge Discovery and Data Mining (KDD'97)},
  pages     = {283--296},
  address   = {Newport Beach, CA, USA},
  publisher = {AAAI Press},
  year      = {1997}
}

@article{zaki2000scalable,
  title   = {Scalable Algorithms for Association Mining},
  author  = {Zaki, Mohammed J.},
  journal = {IEEE Transactions on Knowledge and Data Engineering},
  volume  = {12},
  number  = {3},
  pages   = {372--390},
  year    = {2000},
  doi     = {10.1109/69.846291}
}
```

### APA
- Zaki, M. J., Parthasarathy, S., Ogihara, M., & Li, W. (1997). *New algorithms for fast discovery of association rules*. En **Proceedings of the 3rd International Conference on Knowledge Discovery and Data Mining (KDD'97)** (pp. 283–296). AAAI Press.
- Zaki, M. J. (2000). *Scalable algorithms for association mining*. **IEEE Transactions on Knowledge and Data Engineering, 12**(3), 372–390. https://doi.org/10.1109/69.846291

> *Nota: el algoritmo ECLAT se introdujo en el artículo de KDD'97; el de IEEE TKDE (2000) lo describe y extiende.*

## 3. Tipo de Modelo

| Criterio | Clasificación |
|---|---|
| **Método de aprendizaje** | No supervisado |
| **Por parámetros** | No paramétrico |
| **Datos de aprendizaje** | Offline (batch) — requiere acceso a toda la base antes de comenzar |
| **Resultado del entrenamiento** | Conjunto de **reglas de asociación** + itemsets frecuentes |

Notas:
- **No supervisado** porque no necesita etiquetas; descubre patrones en los datos.
- **No paramétrico** porque no asume una forma funcional ni un número fijo de parámetros a ajustar.
- **Offline** porque la versión clásica necesita acceso a todos los datos; existen variantes online pero son menos comunes.
- ECLAT se destaca por su **eficiencia en I/O** y mejor uso del cache mediante la representación vertical.

## 4. Algoritmo de Entrenamiento

ECLAT es una búsqueda en profundidad (*Depth-First Search*, DFS) sobre un lattice de itemsets. Usa **proyecciones** y **TID-sets** para podar el espacio de búsqueda.

### Pseudocódigo
```text
Entrada:  D (transacciones), min_sup (soporte mínimo)
Salida:   L = conjunto de itemsets frecuentes de todos los tamaños

# Paso 1: Crear representación vertical (TID-sets para ítems únicos)
prefix = ∅
para cada ítem i en I:
    tidset(i) = { j : i ∈ T_j }  # TIDs donde aparece i

# Paso 2: DFS recursivo
Función eclat_recursivo(prefix, tidset_list):
    para cada tidset_i en tidset_list:
        X = prefix ∪ {i}
        soporte(X) = |tidset_i| / |D|
        
        si soporte(X) >= min_sup:
            L = L ∪ {X}  # Guardar itemset frecuente
            
            # Intersectar con ítems posteriores para crear nuevos candidatos
            tidset_list_nuevo = ∅
            para cada tidset_j en tidset_list con j > i:
                tidset_intersección = tidset_i ∩ tidset_j
                soporte_intersección = |tidset_intersección| / |D|
                
                si soporte_intersección >= min_sup:
                    tidset_list_nuevo = tidset_list_nuevo ∪ {tidset_intersección}
            
            eclat_recursivo(X, tidset_list_nuevo)  # Llamada recursiva

eclat_recursivo(∅, [tidset(i) para todo i])
devolver L
```

### Estrategia de búsqueda

El algoritmo utiliza los siguientes principios clave:

1. **Cálculo eficiente del soporte:** $\text{sup}(X \cup Y) = \frac{|\text{TID}(X) \cap \text{TID}(Y)|}{|D|}$
2. **Búsqueda en profundidad (DFS):** exploración recursiva del espacio de itemsets.
3. **Poda agresiva:** elimina candidatos cuando el soporte cae bajo el umbral mínimo.
4. **Mejor uso de memoria caché:** los TID-sets permiten operaciones de intersección rápidas y localizadas.

### Métricas clave

Métricas estándar para medir la calidad de las reglas de asociación:

- **Soporte:**  $\;\;\text{sup}(X) = \dfrac{|\text{TID}(X)|}{|D|}$
- **Confianza:** $\;\;\text{conf}(X \Rightarrow Y) = \dfrac{\text{sup}(X \cup Y)}{\text{sup}(X)}$
- **Lift:** $\;\;\text{lift}(X \Rightarrow Y) = \dfrac{\text{conf}(X \Rightarrow Y)}{\text{sup}(Y)}$

## 5. Supuestos y Restricciones

- **Datos transaccionales categóricos:** los ítems deben ser discretos. Variables numéricas requieren discretización previa.
- **Representación binaria:** cada transacción se modela por la *presencia/ausencia* del ítem (no usa cantidades).
- **Umbrales definidos por el usuario:** `min_support` es un hiperparámetro que condiciona los resultados.
- **Principio de poda:** los TID-sets vacíos indican que no hay itemsets frecuentes en esa rama.
- **Coste de memoria:** puede requerir almacenar muchos TID-sets intermedios en memoria si la base es muy grande o muy densa.
- **Orden lexicográfico:** el desempeño puede depender del orden en que se procesan los ítems.
- **No considera el orden ni el tiempo** entre transacciones (para eso existen variantes como GSP o PrefixSpan).
- **Reglas redundantes:** suele generar muchas reglas similares; requiere post-filtrado para mejorar la interpretabilidad.

## 6. Tests / Métricas de validación

ECLAT se valida con las siguientes **métricas de interés** sobre las reglas descubiertas:

- **Soporte (support)** — frecuencia relativa del itemset.
- **Confianza (confidence)** — probabilidad condicional $P(Y \mid X)$.
- **Lift** — cuánto se desvía la regla de la independencia estadística.
- **Leverage** — diferencia entre la frecuencia observada y la esperada bajo independencia.
- **Conviction** — fortaleza de la implicación; $\infty$ si la regla es perfecta.

Adicionalmente, para evaluar la **eficiencia del algoritmo** (no del modelo):

- **Tiempo de ejecución** — métrica crítica para datasets grandes.
- **Consumo de memoria** — puede ser mayor debido al almacenamiento de TID-sets.
- **Número de candidatos generados** — impacta directamente en la eficiencia.

La calidad práctica también se complementa con análisis cualitativo y validación con datos nuevos.

---
## 7. Implementación práctica

Implementaremos ECLAT desde cero para entender su funcionamiento, y también usaremos **`mlxtend`** para validación de resultados.

### 7.1 Instalación e imports

In [1]:
# Si se ejecuta en Colab o un entorno sin las librerías, descomentar:
%pip install mlxtend pandas

import pandas as pd
from collections import defaultdict
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import association_rules

print('Librerías cargadas correctamente')

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Usuario\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


Librerías cargadas correctamente


### 7.2 Dataset de ejemplo — Cesta de la compra

Utilizamos un dataset de supermercado con transacciones de compra.

In [2]:
transacciones = [
    ['pan', 'leche', 'huevos'],
    ['pan', 'pañales', 'cerveza', 'huevos'],
    ['leche', 'pañales', 'cerveza', 'cola'],
    ['pan', 'leche', 'pañales', 'cerveza'],
    ['pan', 'leche', 'pañales', 'cola'],
    ['pan', 'leche'],
    ['pan', 'cerveza'],
    ['leche', 'pañales'],
    ['pan', 'leche', 'pañales'],
    ['cerveza', 'cola'],
]

print(f'Número de transacciones: {len(transacciones)}')
for i, t in enumerate(transacciones, 1):
    print(f'T{i:>2}: {t}')

Número de transacciones: 10
T 1: ['pan', 'leche', 'huevos']
T 2: ['pan', 'pañales', 'cerveza', 'huevos']
T 3: ['leche', 'pañales', 'cerveza', 'cola']
T 4: ['pan', 'leche', 'pañales', 'cerveza']
T 5: ['pan', 'leche', 'pañales', 'cola']
T 6: ['pan', 'leche']
T 7: ['pan', 'cerveza']
T 8: ['leche', 'pañales']
T 9: ['pan', 'leche', 'pañales']
T10: ['cerveza', 'cola']


### 7.3 Crear representación vertical (TID-sets)

Convertimos de representación horizontal a vertical: cada ítem apunta al conjunto de IDs de transacciones que lo contienen.

In [3]:
# Crear TID-sets para cada ítem
def crear_tidsets_iniciales(transacciones):
    """Retorna un diccionario {ítem: conjunto de TIDs}"""
    tidsets = defaultdict(set)
    for tid, transaccion in enumerate(transacciones):
        for item in transaccion:
            tidsets[item].add(tid)
    return dict(tidsets)

tidsets_iniciales = crear_tidsets_iniciales(transacciones)

print("Representación Vertical (TID-sets):")
for item in sorted(tidsets_iniciales.keys()):
    print(f"  {item:10s}: {tidsets_iniciales[item]}  (sup={len(tidsets_iniciales[item])/len(transacciones):.2f})")

Representación Vertical (TID-sets):
  cerveza   : {1, 2, 3, 6, 9}  (sup=0.50)
  cola      : {9, 2, 4}  (sup=0.30)
  huevos    : {0, 1}  (sup=0.20)
  leche     : {0, 2, 3, 4, 5, 7, 8}  (sup=0.70)
  pan       : {0, 1, 3, 4, 5, 6, 8}  (sup=0.70)
  pañales   : {1, 2, 3, 4, 7, 8}  (sup=0.60)


### 7.4 Implementación del algoritmo ECLAT

In [4]:
def eclat_recursivo(prefix, tidsets_dict, min_sup_count, itemsets_frecuentes, num_transacciones):
    """
    Búsqueda DFS recursiva en ECLAT.
    
    Args:
        prefix: itemset actual (frozenset)
        tidsets_dict: diccionario {ítem: tidset} para candidatos
        min_sup_count: número mínimo de transacciones (min_sup * num_transacciones)
        itemsets_frecuentes: acumulador de resultados
        num_transacciones: total de transacciones
    """
    items = sorted(tidsets_dict.keys())  # Orden lexicográfico para reproducibilidad
    
    for i, item_i in enumerate(items):
        tidset_i = tidsets_dict[item_i]
        sup_i = len(tidset_i)
        
        # Si el itemset es frecuente
        if sup_i >= min_sup_count:
            itemset = prefix | frozenset([item_i])
            itemsets_frecuentes.append((itemset, sup_i / num_transacciones))
            
            # Generar candidatos de nivel k+1 por intersección
            tidsets_nuevo = {}
            for j, item_j in enumerate(items[i+1:], i+1):
                tidset_j = tidsets_dict[item_j]
                tidset_ij = tidset_i & tidset_j  # Intersección
                sup_ij = len(tidset_ij)
                
                if sup_ij >= min_sup_count:
                    tidsets_nuevo[item_j] = tidset_ij
            
            # Llamada recursiva
            if tidsets_nuevo:
                eclat_recursivo(itemset, tidsets_nuevo, min_sup_count, itemsets_frecuentes, num_transacciones)

def eclat(transacciones, min_support):
    """
    Algoritmo ECLAT para mining de itemsets frecuentes.
    
    Args:
        transacciones: lista de listas de ítems
        min_support: umbral de soporte (0.0 a 1.0)
    
    Returns:
        DataFrame con columnas ['itemset', 'support']
    """
    num_transacciones = len(transacciones)
    min_sup_count = max(1, int(min_support * num_transacciones))
    
    tidsets_iniciales = crear_tidsets_iniciales(transacciones)
    itemsets_frecuentes = []
    
    # DFS recursivo
    eclat_recursivo(frozenset(), tidsets_iniciales, min_sup_count, itemsets_frecuentes, num_transacciones)
    
    # Convertir a DataFrame
    df = pd.DataFrame(itemsets_frecuentes, columns=['itemsets', 'support'])
    df = df[df['itemsets'].apply(len) > 0]  # Excluir itemset vacío si existe
    df = df.sort_values('support', ascending=False).reset_index(drop=True)
    
    return df

print('Funciones ECLAT definidas correctamente')

Funciones ECLAT definidas correctamente


### 7.5 Ejecutar ECLAT y obtener itemsets frecuentes

In [5]:
min_sup = 0.3
itemsets_eclat = eclat(transacciones, min_sup)

# Formatear para visualización
itemsets_eclat_display = itemsets_eclat.copy()
itemsets_eclat_display['itemsets'] = itemsets_eclat_display['itemsets'].apply(lambda x: ', '.join(sorted(x)))

print(f"Itemsets frecuentes (min_support = {min_sup}):")
print(itemsets_eclat_display)

Itemsets frecuentes (min_support = 0.3):
               itemsets  support
0                 leche      0.7
1                   pan      0.7
2               pañales      0.6
3        leche, pañales      0.5
4            leche, pan      0.5
5               cerveza      0.5
6          pan, pañales      0.4
7      cerveza, pañales      0.3
8          cerveza, pan      0.3
9   leche, pan, pañales      0.3
10                 cola      0.3


### 7.6 Generación de reglas de asociación usando mlxtend

In [6]:
# Convertir al formato que mlxtend espera
te = TransactionEncoder()
te_array = te.fit(transacciones).transform(transacciones)
df_te = pd.DataFrame(te_array, columns=te.columns_)

# Para usar association_rules, necesitamos convertir nuestro DataFrame de itemsets
# a un formato compatible con mlxtend
itemsets_eclat_mlxtend = itemsets_eclat.copy()
itemsets_eclat_mlxtend['itemsets'] = itemsets_eclat_mlxtend['itemsets'].apply(frozenset)

# Generar reglas con mlxtend
from mlxtend.frequent_patterns import association_rules

reglas = association_rules(itemsets_eclat_mlxtend, metric='confidence', min_threshold=0.6)

if len(reglas) > 0:
    columnas = ['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage', 'conviction']
    reglas = reglas[columnas].sort_values('lift', ascending=False).reset_index(drop=True)
    print("\nReglas de Asociación (min_confidence = 0.6):")
    print(reglas)
else:
    print("\nNo se generaron reglas con los umbrales especificados.")


Reglas de Asociación (min_confidence = 0.6):
                   antecedents           consequents  support  confidence  \
0         frozenset({pañales})    frozenset({leche})      0.5    0.833333   
1           frozenset({leche})  frozenset({pañales})      0.5    0.714286   
2    frozenset({pañales, pan})    frozenset({leche})      0.3    0.750000   
3           frozenset({leche})      frozenset({pan})      0.5    0.714286   
4             frozenset({pan})    frozenset({leche})      0.5    0.714286   
5         frozenset({cerveza})  frozenset({pañales})      0.3    0.600000   
6      frozenset({leche, pan})  frozenset({pañales})      0.3    0.600000   
7         frozenset({pañales})      frozenset({pan})      0.4    0.666667   
8         frozenset({cerveza})      frozenset({pan})      0.3    0.600000   
9  frozenset({leche, pañales})      frozenset({pan})      0.3    0.600000   

       lift  leverage  conviction  
0  1.190476      0.08        1.80  
1  1.190476      0.08        1.40 

### 7.7 Interpretación

In [7]:
print("\n" + "="*70)
print("ANÁLISIS DE RESULTADOS")
print("="*70)

print(f"\nDataset: {len(transacciones)} transacciones, min_support={min_sup}")
print(f"Itemsets frecuentes encontrados: {len(itemsets_eclat)}")

print("\nCaracterísticas del algoritmo ECLAT:")
print("  • Representación vertical con TID-sets")
print("  • Búsqueda recursiva en profundidad (DFS)")
print("  • Cálculo eficiente de soporte mediante intersecciones de conjuntos")
print("  • Mejor escalabilidad en datasets densos")
print("  • Menores requerimientos de I/O en bases grandes")


ANÁLISIS DE RESULTADOS

Dataset: 10 transacciones, min_support=0.3
Itemsets frecuentes encontrados: 11

Características del algoritmo ECLAT:
  • Representación vertical con TID-sets
  • Búsqueda recursiva en profundidad (DFS)
  • Cálculo eficiente de soporte mediante intersecciones de conjuntos
  • Mejor escalabilidad en datasets densos
  • Menores requerimientos de I/O en bases grandes


### 7.7 Top reglas por Lift

In [8]:
if len(reglas) > 0:
    print('\nTop 5 reglas por LIFT:')
    for idx, (_, r) in enumerate(reglas.head(5).iterrows(), 1):
        ant = ', '.join(sorted(r['antecedents']))
        con = ', '.join(sorted(r['consequents']))
        print(f'  {idx}. {{{ant}}}  =>  {{{con}}}')
        print(f'     sup={r["support"]:.3f}  conf={r["confidence"]:.3f}  lift={r["lift"]:.3f}\n')
else:
    print("No hay reglas para mostrar.")


Top 5 reglas por LIFT:
  1. {pañales}  =>  {leche}
     sup=0.500  conf=0.833  lift=1.190

  2. {leche}  =>  {pañales}
     sup=0.500  conf=0.714  lift=1.190

  3. {pan, pañales}  =>  {leche}
     sup=0.300  conf=0.750  lift=1.071

  4. {leche}  =>  {pan}
     sup=0.500  conf=0.714  lift=1.020

  5. {pan}  =>  {leche}
     sup=0.500  conf=0.714  lift=1.020



## 8. Conclusión

- ECLAT es un algoritmo **no supervisado y no paramétrico** que descubre **reglas de asociación** en datos transaccionales mediante una búsqueda recursiva en profundidad.
- Su fortaleza radica en la **representación vertical (TID-sets)** y el cálculo eficiente del soporte mediante intersecciones de conjuntos.
- La estrategia DFS permite **descartar ramas del árbol de búsqueda temprano**, mejorando significativamente la eficiencia computacional.
- ECLAT generalmente requiere **menos pasadas sobre la base de datos** y consume menos I/O comparado con enfoques por niveles.
- Aunque puede consumir más memoria temporal almacenando TID-sets intermedios, ofrece un balance favorable entre velocidad y escalabilidad.
- Variantes modernas como **FP-Growth** y **dEclat** mejoran aún más la eficiencia mediante estructuras de datos especializadas o procesamiento distribuido.